In [400]:
import pandas as pd
import numpy as np
from datetime import datetime

In [401]:
# Set max display
pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", None)

## **Read Data & Config**

In [ ]:
# Load datasets
customers = pd.read_csv('../data/customers.csv')
geography = pd.read_csv('../data/geography.csv')
inventory = pd.read_csv('../data/inventory.csv')
order_items = pd.read_csv('../data/order_items.csv')
orders = pd.read_csv('../data/orders.csv')
payments = pd.read_csv('../data/payments.csv')
products = pd.read_csv('../data/products.csv')
promotions = pd.read_csv('../data/promotions.csv')
returns = pd.read_csv('../data/returns.csv')
reviews = pd.read_csv('../data/reviews.csv')
sales = pd.read_csv('../data/sales.csv')
shipments = pd.read_csv('../data/shipments.csv')
web_traffic = pd.read_csv('../data/web_traffic.csv')

/var/folders/p9/dtph93rn4j546kzjl41m5yt00000gn/T/ipykernel_87322/2328962422.py:5: DtypeWarning: Columns (0: promo_id_2) have mixed types. Specify dtype option on import or set low_memory=False.
  order_items = pd.read_csv('data/order_items.csv')


In [403]:
# Config

# Snapshot date: the "current date" used to calculate Recency
# Set to None to automatically use max(order_date) in the data
SNAPSHOT_DATE = None 

# Churn threshold (for customers who had only one order)
ONE_TIME_FALLBACK_DAYS = 90
Z_SCORE = 2.0   # ~95% confidence

## **Preprocess**

In [404]:
orders["order_status"].value_counts()

order_status
delivered    516716
cancelled     59462
returned      36142
shipped       13773
paid          13577
created        7275
Name: count, dtype: int64

In [405]:
# Convert datetime columns
orders["order_date"] = pd.to_datetime(
    orders["order_date"],
    errors="coerce"
)

customers["signup_date"] = pd.to_datetime(
    customers["signup_date"],
    errors="coerce"
)

returns["return_date"] = pd.to_datetime(
    returns["return_date"],
    errors="coerce"
   )

# Filter valid orders 
valid_statuses = ["delivered", "shipped", "returned", "paid"]

orders_valid = orders[
    orders["order_status"].str.lower().isin(valid_statuses)
].copy()

print("Valid orders:", orders_valid.shape)

Valid orders: (580208, 8)


In [406]:
# Get snapshot date
if SNAPSHOT_DATE:
    snapshot = pd.to_datetime(SNAPSHOT_DATE)
else:
    snapshot = orders["order_date"].max()

print("Snapshot date:", snapshot)

Snapshot date: 2022-12-31 00:00:00


## **Build LRFMD Features**

### **Revenue Features**

In [407]:
items = order_items.copy()

In [408]:
# Merge products
if not products.empty:
    items = items.merge(
        products[["product_id", "category", "segment"]],
        on="product_id",
        how="left"
    )

# Revenue per item
items["line_revenue"] = (
    items["unit_price"] * items["quantity"]
)

# Revenue per order
rev_per_order = (
    items.groupby("order_id")["line_revenue"]
    .sum()
    .reset_index()
    .rename(columns={"line_revenue": "order_revenue"})
)

In [409]:
items.head()

,order_id,product_id,quantity,unit_price,discount_amount,promo_id,promo_id_2,category,segment,line_revenue
0,1,2400,7,1138.22,0.0,NaN,NaN,GenZ,Trendy,7967.54
1,2,609,7,10166.25,0.0,NaN,NaN,Streetwear,Everyday,71163.75
2,3,396,3,11220.33,0.0,NaN,NaN,Streetwear,Balanced,33660.99
3,4,635,5,10639.25,0.0,NaN,NaN,Streetwear,Everyday,53196.25
4,6,1935,1,1597.84,0.0,NaN,NaN,Outdoor,Activewear,1597.84


In [410]:
rev_per_order.head()

,order_id,order_revenue
0,1,7967.54
1,2,71163.75
2,3,33660.99
3,4,53196.25
4,6,1597.84


In [411]:
# Refund per order
if not returns.empty and "refund_amount" in returns.columns:
    return_by_order = (
        returns.groupby("order_id")["refund_amount"]
        .sum()
        .reset_index()
        .rename(columns={
            "refund_amount": "total_refund_raw"
        })
    )

else:
    return_by_order = pd.DataFrame(
        columns=["order_id", "total_refund_raw"]
    )

return_by_order.head()

,order_id,total_refund_raw
0,2,52458.01
1,32,5141.37
2,35,5315.95
3,47,8234.51
4,59,10086.33


In [412]:
# Merge revenue + refund
orders_enriched = orders_valid.merge(
    rev_per_order,
    on="order_id",
    how="left"
)

orders_enriched = orders_enriched.merge(
    return_by_order,
    on="order_id",
    how="left"
)

# orders_enriched["order_revenue"] = (
#     orders_enriched["order_revenue"].fillna(0)
# )

orders_enriched["total_refund_raw"] = (
    orders_enriched["total_refund_raw"].fillna(0)
)

# Cap refund
orders_enriched["total_refund"] = (
    orders_enriched[
        ["total_refund_raw", "order_revenue"]
    ].min(axis=1)
)

# Refund anomaly flag
orders_enriched["refund_anomaly_flag"] = (
    orders_enriched["total_refund_raw"]
    > orders_enriched["order_revenue"]
).astype(int)

# Net revenue
orders_enriched["net_revenue"] = (
    orders_enriched["order_revenue"]
    - orders_enriched["total_refund"]
).clip(lower=0)

### **Diversity Feature (D)**

In [413]:
items_with_cust = items.merge(
    orders_valid[["order_id", "customer_id"]],
    on="order_id",
    how="left"
)
items_with_cust.head()

,order_id,product_id,quantity,unit_price,discount_amount,promo_id,promo_id_2,category,segment,line_revenue,customer_id
0,1,2400,7,1138.22,0.0,NaN,NaN,GenZ,Trendy,7967.54,58578.0
1,2,609,7,10166.25,0.0,NaN,NaN,Streetwear,Everyday,71163.75,58621.0
2,3,396,3,11220.33,0.0,NaN,NaN,Streetwear,Balanced,33660.99,58811.0
3,4,635,5,10639.25,0.0,NaN,NaN,Streetwear,Everyday,53196.25,59453.0
4,6,1935,1,1597.84,0.0,NaN,NaN,Outdoor,Activewear,1597.84,57821.0


In [414]:
diversity = (
    items_with_cust.groupby("customer_id")
    .agg(
        n_categories=("category", "nunique"),   # Number of unique categories
        n_segments=("segment", "nunique"),  # Number of unique segments
        n_products=("product_id", "nunique")    # Number of unique products
    )
    .reset_index()
)

diversity["D"] = diversity["n_categories"]
diversity.head()

,customer_id,n_categories,n_segments,n_products,D
0,1.0,2,3,7,2
1,2.0,1,2,2,1
2,3.0,2,2,2,2
3,4.0,1,1,1,1
4,5.0,3,4,5,3


### **LRFM Features**

In [415]:
orders_enriched.head()

,order_id,order_date,customer_id,zip,order_status,payment_method,device_type,order_source,order_revenue,total_refund_raw,total_refund,refund_anomaly_flag,net_revenue
0,1,2012-07-04,58578,1109,delivered,credit_card,desktop,paid_search,7967.54,0.00,0.00,0,7967.54
1,2,2012-07-04,58621,1330,returned,cod,mobile,paid_search,71163.75,52458.01,52458.01,0,18705.74
2,3,2012-07-04,58811,1473,delivered,credit_card,desktop,direct,33660.99,0.00,0.00,0,33660.99
3,4,2012-07-04,59453,2360,delivered,credit_card,desktop,referral,53196.25,0.00,0.00,0,53196.25
4,6,2012-07-06,57821,2886,delivered,paypal,mobile,email_campaign,1597.84,0.00,0.00,0,1597.84


In [416]:
# Frequency, Monetary
lrfm = (
    orders_enriched.groupby("customer_id")
    .agg(
        first_order_date=("order_date", "min"),
        last_order_date=("order_date", "max"),
        F=("order_id", "count"),
        M=("net_revenue", "sum")
    )
    .reset_index()
)

# Recency
lrfm["R"] = (snapshot - lrfm["last_order_date"]).dt.days

# Length
lrfm["L"] = (lrfm["last_order_date"] - lrfm["first_order_date"]).dt.days

In [417]:
# Merge diversity
lrfm = lrfm.merge(
    diversity[
        [
            "customer_id",
            "D",
            "n_categories",
            "n_segments",
            "n_products"
        ]
    ],
    on="customer_id",
    how="left"
)

lrfm[["D", "n_categories", "n_segments", "n_products"]] = (lrfm[["D", "n_categories", "n_segments", "n_products"]].astype(int))
lrfm.head()

,customer_id,first_order_date,last_order_date,F,M,R,L,D,n_categories,n_segments,n_products
0,1,2012-07-25,2021-04-24,6,141404.83,616,3195,2,2,3,7
1,2,2013-09-20,2022-07-06,2,146787.34,178,3211,1,1,2,2
2,3,2012-08-27,2012-11-24,2,12001.55,3689,89,2,2,2,2
3,4,2020-06-28,2020-06-28,1,13340.32,916,0,1,1,1,1
4,5,2012-08-09,2019-03-27,5,69075.34,1375,2421,3,3,4,5


### **IPT(Inter-purchase time)**

In [418]:
# Only calculate for customers with >= 2 orders
order_dates = (
    orders_valid[["customer_id", "order_date"]]
    .sort_values(["customer_id", "order_date"])
    .copy()
)

order_dates["prev_order_date"] = (order_dates.groupby("customer_id")["order_date"].shift(1))
order_dates["ipt_days"] = (order_dates["order_date"] - order_dates["prev_order_date"]).dt.days

ipt_stats = (
    order_dates.dropna(subset=["ipt_days"])
    .groupby("customer_id")["ipt_days"]
    .agg(
        ipt_mean="mean",
        ipt_std="std",
        ipt_n="count"
    )
    .reset_index()
)

ipt_stats["ipt_std"] = ipt_stats["ipt_std"].fillna(0) # If only 2 orders, std = 0

lrfm = lrfm.merge(
    ipt_stats,
    on="customer_id",
    how="left"
)

# For one-time customers, IPT cannot be calculated → fillna after merge
lrfm["ipt_std"] = lrfm["ipt_std"].fillna(0)
lrfm["ipt_n"] = lrfm["ipt_n"].fillna(0).astype(int)
# keep ipt_mean as NaN to distinguish "not enough data" vs "actual IPT"
# churn_threshold will use fallback (ONE_TIME_FALLBACK_DAYS = 90) when ipt_mean is NaN

lrfm.head()

,customer_id,first_order_date,last_order_date,F,M,R,L,D,n_categories,n_segments,n_products,ipt_mean,ipt_std,ipt_n
0,1,2012-07-25,2021-04-24,6,141404.83,616,3195,2,2,3,7,639.00,250.456583,5
1,2,2013-09-20,2022-07-06,2,146787.34,178,3211,1,1,2,2,3211.00,0.000000,1
2,3,2012-08-27,2012-11-24,2,12001.55,3689,89,2,2,2,2,89.00,0.000000,1
3,4,2020-06-28,2020-06-28,1,13340.32,916,0,1,1,1,1,NaN,0.000000,0
4,5,2012-08-09,2019-03-27,5,69075.34,1375,2421,3,3,4,5,605.25,528.470986,4


## **Derived Features**

In [419]:
df = lrfm.copy()

In [420]:
# Average order value
df["avg_order_value"] = df["M"] / df["F"]

# # Purchase rate
# df["tenure_months"] = (df["L"] / 30)
# df["purchase_rate_per_month"] = df["F"] / df["tenure_months"]

# Customer age
df = df.merge(customers[["customer_id", "signup_date"]], on="customer_id", how="left")
df["customer_age_days"] = (snapshot - pd.to_datetime(df["signup_date"], errors="coerce")).dt.days

df.head()

,customer_id,first_order_date,last_order_date,F,M,R,L,D,n_categories,n_segments,n_products,ipt_mean,ipt_std,ipt_n,avg_order_value,signup_date,customer_age_days
0,1,2012-07-25,2021-04-24,6,141404.83,616,3195,2,2,3,7,639.00,250.456583,5,23567.471667,2021-12-30,366
1,2,2013-09-20,2022-07-06,2,146787.34,178,3211,1,1,2,2,3211.00,0.000000,1,73393.670000,2013-12-27,3291
2,3,2012-08-27,2012-11-24,2,12001.55,3689,89,2,2,2,2,89.00,0.000000,1,6000.775000,2018-07-24,1621
3,4,2020-06-28,2020-06-28,1,13340.32,916,0,1,1,1,1,NaN,0.000000,0,13340.320000,2017-11-29,1858
4,5,2012-08-09,2019-03-27,5,69075.34,1375,2421,3,3,4,5,605.25,528.470986,4,13815.068000,2022-09-23,99


In [421]:
# Return features
# if not returns.empty:
ret_enrich = returns.merge(
    orders_valid[
        ["order_id", "customer_id"]
    ],
    on="order_id",
    how="left"
)

ret_enrich = ret_enrich.merge(
    orders_enriched[
        [
            "order_id",
            "refund_anomaly_flag"
        ]
    ],
    on="order_id",
    how="left"
)

return_stats = (
    ret_enrich.groupby("customer_id")
    .agg(
        n_returns=("return_id", "count"),
        total_refund_amt=("refund_amount", "sum")
    )
    .reset_index()
)

df = df.merge(return_stats, on="customer_id", how="left")

df["n_returns"] = df["n_returns"].fillna(0).astype(int)
df["total_refund_amt"] = df["total_refund_amt"].fillna(0.0)
df["return_rate"] = df["n_returns"] / df["F"].replace(0, np.nan)

In [422]:
df.head()

,customer_id,first_order_date,last_order_date,F,M,R,L,D,n_categories,n_segments,n_products,ipt_mean,ipt_std,ipt_n,avg_order_value,signup_date,customer_age_days,n_returns,total_refund_amt,return_rate
0,1,2012-07-25,2021-04-24,6,141404.83,616,3195,2,2,3,7,639.00,250.456583,5,23567.471667,2021-12-30,366,1,1398.64,0.166667
1,2,2013-09-20,2022-07-06,2,146787.34,178,3211,1,1,2,2,3211.00,0.000000,1,73393.670000,2013-12-27,3291,0,0.00,0.000000
2,3,2012-08-27,2012-11-24,2,12001.55,3689,89,2,2,2,2,89.00,0.000000,1,6000.775000,2018-07-24,1621,0,0.00,0.000000
3,4,2020-06-28,2020-06-28,1,13340.32,916,0,1,1,1,1,NaN,0.000000,0,13340.320000,2017-11-29,1858,0,0.00,0.000000
4,5,2012-08-09,2019-03-27,5,69075.34,1375,2421,3,3,4,5,605.25,528.470986,4,13815.068000,2022-09-23,99,0,0.00,0.000000


In [423]:
# Promotion features
items_cust = order_items.merge(
    orders_valid[
        ["order_id", "customer_id"]
    ],
    on="order_id",
    how="left"
)

promo_stats = (
    items_cust.groupby("customer_id")
    .agg(
        n_promo_used=(
            "promo_id",
            lambda x: x.notna().sum()
        ),
        total_discount_amt=(
            "discount_amount",
            "sum"
        )
    )
    .reset_index()
)

df = df.merge(promo_stats, on="customer_id", how="left")

df["n_promo_used"] = df["n_promo_used"].fillna(0).astype(int)
df["total_discount_amt"] = df["total_discount_amt"].fillna(0.0)
df["promo_dependency"] = df["n_promo_used"] / df["F"].replace(0, np.nan)

## **Customer Profile**

In [424]:
profile_cols = ["customer_id", "zip", "city", "gender", "age_group", "acquisition_channel"]
profile_cols = [c for c in profile_cols if c in customers.columns]
df = df.merge(customers[profile_cols], on="customer_id", how="left")

geo_cols = ["zip", "region", "district"]
geo_cols = [c for c in geo_cols if c in geography.columns]
df = df.merge(geography[geo_cols], on="zip", how="left")

In [425]:
df[["customer_id", "zip", "city", "gender", "age_group", "acquisition_channel", "zip", "region", "district"]].head()

,customer_id,zip,city,gender,age_group,acquisition_channel,zip,region,district
0,1,15201,Hai Phong,Female,35-44,social_media,15201,East,District #13
1,2,15201,Hai Phong,Female,45-54,email_campaign,15201,East,District #13
2,3,15201,Hai Phong,Female,18-24,organic_search,15201,East,District #13
3,4,15201,Hai Phong,Male,35-44,referral,15201,East,District #13
4,5,15201,Hai Phong,Male,55+,organic_search,15201,East,District #13


## **Churn Label**

In [426]:
df["churn_threshold_days"] = np.where(
    df["ipt_mean"].notna(),
    df["ipt_mean"] + Z_SCORE * df["ipt_std"],
    ONE_TIME_FALLBACK_DAYS
).round(1)

# Minimum threshold = 7 days
df["churn_threshold_days"] = df["churn_threshold_days"].clip(lower=7)

# Churn label
df["is_churn"] = (df["R"] > df["churn_threshold_days"]).astype(int)

df["ipt_mean"] = df["ipt_mean"].round(2)
df["ipt_std"] = df["ipt_std"].round(2)

In [427]:
df[["customer_id", "churn_threshold_days", "is_churn"]].isnull().sum()

customer_id             0
churn_threshold_days    0
is_churn                0
dtype: int64

In [428]:
df[["customer_id", "churn_threshold_days", "is_churn"]].head()

,customer_id,churn_threshold_days,is_churn
0,1,1139.9,0
1,2,3211.0,0
2,3,89.0,1
3,4,90.0,1
4,5,1662.2,0


## **Final df**

In [429]:
df.columns

Index(['customer_id', 'first_order_date', 'last_order_date', 'F', 'M', 'R',
       'L', 'D', 'n_categories', 'n_segments', 'n_products', 'ipt_mean',
       'ipt_std', 'ipt_n', 'avg_order_value', 'signup_date',
       'customer_age_days', 'n_returns', 'total_refund_amt', 'return_rate',
       'n_promo_used', 'total_discount_amt', 'promo_dependency', 'zip', 'city',
       'gender', 'age_group', 'acquisition_channel', 'region', 'district',
       'churn_threshold_days', 'is_churn'],
      dtype='str')

In [430]:
COLUMN_ORDER = [
    # ID
    "customer_id",
    # Profile
    "gender", "age_group", "acquisition_channel",
    "city", "region", "district", "zip",
    "signup_date", "customer_age_days",
    # LRFMD
    "first_order_date", "last_order_date",
    "L",   # Length
    "R",   # Recency
    "F",   # Frequency
    "M",   # Monetary (net)
    "D",   # Diversity (n_categories)
    "n_categories", "n_segments", "n_products",
    # IPT (Inter-Purchase Time)
    "ipt_n", "ipt_mean", "ipt_std",
    # Derived
    "avg_order_value",
    "n_returns", "total_refund_amt", "return_rate",
    "n_promo_used", "total_discount_amt", "promo_dependency",
    # Churn
    "churn_threshold_days",   # μ_IPT + 2σ_IPT 
    "is_churn",
]

In [431]:
existing = [c for c in COLUMN_ORDER if c in df.columns]
extra = [c for c in df.columns if c not in existing]
df = df[existing + extra]

In [432]:
# Final selected columns
FINAL_COLUMNS = [
    # Identity
    "customer_id",
    # Profile
    "gender", "age_group", "acquisition_channel",
    "city", "region", "zip",
    "signup_date", "customer_age_days",
    # LRFMD
    "L", "R", "F", "M", "D",
    "n_segments", "n_products",
    # Derived
    "avg_order_value",
    "return_rate",
    "total_discount_amt", "promo_dependency",
    # Churn label
    "is_churn",
]

df = df[FINAL_COLUMNS].copy()

In [433]:
df.head()

,customer_id,gender,age_group,acquisition_channel,city,region,zip,signup_date,customer_age_days,L,R,F,M,D,n_segments,n_products,avg_order_value,return_rate,total_discount_amt,promo_dependency,is_churn
0,1,Female,35-44,social_media,Hai Phong,East,15201,2021-12-30,366,3195,616,6,141404.83,2,3,7,23567.471667,0.166667,0.00,0.0,0
1,2,Female,45-54,email_campaign,Hai Phong,East,15201,2013-12-27,3291,3211,178,2,146787.34,1,2,2,73393.670000,0.000000,20532.05,1.0,0
2,3,Female,18-24,organic_search,Hai Phong,East,15201,2018-07-24,1621,89,3689,2,12001.55,2,2,2,6000.775000,0.000000,0.00,0.0,1
3,4,Male,35-44,referral,Hai Phong,East,15201,2017-11-29,1858,0,916,1,13340.32,1,1,1,13340.320000,0.000000,2401.26,1.0,1
4,5,Male,55+,organic_search,Hai Phong,East,15201,2022-09-23,99,2421,1375,5,69075.34,3,4,5,13815.068000,0.000000,4895.48,0.4,0


In [434]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 87839 entries, 0 to 87838
Data columns (total 21 columns):
 #   Column               Non-Null Count  Dtype         
---  ------               --------------  -----         
 0   customer_id          87839 non-null  int64         
 1   gender               87839 non-null  str           
 2   age_group            87839 non-null  str           
 3   acquisition_channel  87839 non-null  str           
 4   city                 87839 non-null  str           
 5   region               87839 non-null  str           
 6   zip                  87839 non-null  int64         
 7   signup_date          87839 non-null  datetime64[us]
 8   customer_age_days    87839 non-null  int64         
 9   L                    87839 non-null  int64         
 10  R                    87839 non-null  int64         
 11  F                    87839 non-null  int64         
 12  M                    87839 non-null  float64       
 13  D                    87839 non-null  int64